# Exploratory Pitch Stance Pipeline

This notebook explores a frame-by-frame pipeline for pitch-by-pitch catcher stance analytics.

Goals:
- identify the set-stance window inside each pitch clip
- improve catcher selection and reject batter/umpire swaps
- classify stance per frame and aggregate to a pitch label
- filter bad broadcast angles and normalize coordinates

The production code already contains a strong catcher selector in `src/catcher_detection/detector.py`. This notebook builds on that logic rather than replacing it.

In [ ]:
from pathlib import Path

import pandas as pd

from research.scripts.pitch_stance_research import (
    BASEBALLCV_AVAILABLE,
    analyze_directory,
    event_anchor_window,
    majority_vote,
    rolling_majority_vote,
)

video_dir = Path('data/examples/duke-2026-04-21-liberty-sample/downloads')
summaries = analyze_directory(video_dir)
df = pd.json_normalize(summaries)

BASEBALLCV_AVAILABLE

## BaseballCV / RF-DETR Option Space

Official BaseballCV documentation shows the following baseball-specific aliases that matter for this problem:

- `ball_tracking.pt`: ball trajectory and release/contact anchoring
- `glove_tracking.pt`: glove, ball, home plate, and pitcher rubber tracking
- `pitcher_hitter_catcher.pt`: coarse broadcast triage for the three main people in frame
- `rfdetr_glove_tracking`: RF-DETR version of glove / ball / plate / rubber tracking

That maps cleanly to the three research roles we need:
1. anchor the critical pitch event
2. reject person swaps using spatial context
3. optionally tighten broadcast detection before pose extraction

Primary sources:
- BaseballCV GitHub: https://github.com/BaseballCV/BaseballCV
- RF-DETR GitHub: https://github.com/roboflow/rf-detr


In [ ]:
from research.scripts.pitch_stance_research import option_matrix

option_df = pd.DataFrame(option_matrix())
option_df


## Repository Audit Findings

Relevant existing pieces:
- `src/catcher_detection/detector.py`: plate-area ROI gating, catcher-vs-batter/umpire rejection, score-based abstention
- `src/curator/features.py`: YOLO pose streaming, catcher normalization, fixed-length keypoint export
- `src/stance_pipeline/overlay.py`: frame-by-frame pose rendering during replay
- `src/stance_pipeline/model.py`: MLP stance classifier for pitch-level inference

The current pipeline already solves candidate selection fairly well. The main research gap is temporal: which frames inside a clip should actually contribute to the final pitch stance label?

## Working Hypothesis

The cleanest solution is a hybrid pipeline:

- keep the current pose-based catcher detector as the default spatial gate
- use ball or glove tracking as a pitch-event anchor when available
- search for a low-motion `set stance` window when event anchors are missing
- classify each frame inside the window and aggregate by majority vote

This avoids overfitting the whole clip to the pre-pitch stance while still giving a deterministic fallback when BaseballCV or RF-DETR assets are unavailable.

## Objective A: Set Stance Window Identification

Measured on the five checked-in sample clips at `data/examples/duke-2026-04-21-liberty-sample/downloads/`, the catcher detector finds a stable low-motion segment later in each clip.

| Clip | Duration | Valid detections | First valid frame | Best 1.5s window |
| --- | ---: | ---: | ---: | --- |
| `...356-369.mp4` | 13.63s | 126 | 214 | 214-264 |
| `...378-388.mp4` | 10.60s | 79 | 137 | 145-223 |
| `...395-409.mp4` | 14.58s | 99 | 204 | 204-255 |
| `...428-444.mp4` | 16.55s | 245 | 95 | 163-211 |
| `...449-462.mp4` | 13.53s | 242 | 0 | 226-270 |

Interpretation:
- the useful stance window is usually not the entire clip
- low-motion detection windows tend to appear after the catcher first enters frame
- the best default is a hybrid strategy: anchor to a pitch event when available, otherwise fall back to a low-motion window search

A practical default is `1.0-1.5s` of frames immediately before the anchor event or before the onset of catcher motion.

In [ ]:
# Reproduce the sample summary table when running interactively.
cols = [
    'video',
    'duration_s',
    'valid_detections',
    'first_valid_frame',
    'last_valid_frame',
    'best_window',
]
df[cols]


## Objective B: Catcher Detection and Disambiguation

The current detector is already doing meaningful work:

- plate-area ROI gating keeps the catcher near the expected home-plate zone
- invalid zones reject dugout and edge-of-frame clutter
- lower-body geometry rejects upright pitcher-like or batter-like candidates
- anchor-distance scoring prefers a catcher-shaped lower-center pose over a generic crouch

On the sample Duke at Liberty clips, the dominant rejection reasons were pitcher-like low/tall geometry, invalid-zone overlap, narrow stance, and hips too far from the catcher anchor. That is the right failure mode for this task because it prevents swapping the catcher with nearby players.

## Objectives B-D: Detection, Classification, and Camera Quality

Objective B, catcher disambiguation:
- use the current plate-anchor ROI and invalid-zone filters as the first line of defense
- keep rejecting tall/low pitcher-like boxes, narrow stances, and near-edge dugout clutter
- add an external detector such as BaseballCV `pitcher_hitter_catcher.pt` only as an optional candidate generator

Objective C, stance classification:
- classify each valid frame with geometric features or a lightweight model
- aggregate over the selected window with majority vote or a rolling median on per-frame labels
- return one pitch-level stance plus confidence and quality metadata

Objective D, camera filtering and normalization:
- reject clips with too many failed frames or unstable plate-anchor geometry
- normalize keypoints by torso or box scale before any per-frame classifier
- use camera-quality flags so bad broadcast angles never silently become stance labels

## Prompt Framing for the Full Problem

The full system should answer one question per pitch clip: what was the catcher doing in the stationary set-stance before the pitch?

That means the implementation should optimize for three things in order:

1. correct temporal window selection
2. correct catcher identity inside the window
3. stable pitch-level stance aggregation

If any step is uncertain, the pipeline should prefer abstaining or marking the clip low-confidence rather than forcing a stance label from post-pitch motion.

In [ ]:
# Example temporal aggregation helpers.
labels = ['Squat', 'Squat', None, 'RKD', 'RKD', 'RKD']
majority_vote(labels), rolling_majority_vote(labels, window=3), event_anchor_window(anchor_frame_idx=240, fps=60.0, pre_seconds=1.5)


## End-to-End Production Shape

The production path should look like this:

1. decode the clip frame-by-frame
2. run catcher detection and spatial gating on each frame
3. choose a set-stance window from event anchoring or low-motion scoring
4. classify each frame inside that window as `Squat`, `LKD`, or `RKD`
5. aggregate the frame labels into a single pitch result
6. emit the pitch result plus quality metadata and a replay overlay

The spec for this flow is captured in `research/specs/pitch_stance_pipeline_spec.md`.